# 1. モデル構成の比較（FairVLMed）

ImageNet初期化・1-stage ERMで、ResNet-50・Spatial LoRA・HyperAdaptを3 seedずつ比べる。
CheXpertのM01・M03・M07と同じレシピで、データだけを替えたときに属性条件付きモデルとResNetの差が
test標本の雑音より大きくなるかを見る。評価軸はFairCLIPの4軸とage_group_65で、主軸はrace。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.archive.fairvlmed_architecture_erm.comparison import (  # noqa: E402
    AXES,
    MODELS,
    evaluate_all,
    paired_bootstrap,
    plot_group_metric,
    plot_paired_deltas,
    plot_seed_points,
    prediction_command,
    read_scores,
    seed_deltas,
    select_runs,
)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402

plt.style.use(STYLE_SHEET)
PACKAGE = study_dir("fairvlmed_architecture_erm")
TOPIC = "01_architecture_comparison"
RESULTS = PACKAGE / "results" / TOPIC
FIGURES = PACKAGE / "figures" / TOPIC
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
N_BOOT = 1000

## 入力

モデル・seedごとに成功runを一意に選び、best `val/auroc` checkpointと学習時のtrain/val CSVをSHA256で照合する。
testは全runで同じCSVであることを確かめ、予測cacheの行順・targetを検算する。
cacheが欠けていれば、表示されるコマンドをrepo rootで実行する（推論GPUは自動で使わない）。

In [ ]:
provenance, frame, directories = select_runs(PACKAGE)
provenance.to_csv(RESULTS / "provenance.csv", index=False)
print(prediction_command(PACKAGE, directories) or "全予測cacheが有効")
scores = read_scores(PACKAGE, directories, frame)

## 評価集合

主比較は配布testの全2,000件。属性の欠損（ethnicity・languageの`unknown`）はその軸の群からだけ外す。
languageはspanish 31件・other 11件で、最小セル20件を満たすのはenglishの全セルとspanishの陽性側だけになる。

In [ ]:
PRIMARY = np.ones(len(frame), bool)
print(f"主比較: {PRIMARY.sum():,}件（陽性 {frame['target'].sum()}）")

## 全体性能

閾値0.5（argmax）での指標。点がseed、横棒が3 seed平均。seed間のばらつきは学習の揺れで、test標本の雑音を含まない
（後段のpaired bootstrapで見る）。

In [ ]:
summary, groups = evaluate_all(scores, frame, PRIMARY)
summary.to_csv(RESULTS / "metrics_by_seed.csv", index=False)
groups.to_csv(RESULTS / "groups_by_seed.csv", index=False)
summary.groupby("model", sort=False).agg(["mean", "std"]).to_csv(RESULTS / "metrics_mean_std.csv")
overall = {
    "overall/auroc": "AUROC",
    "overall/bacc": "BACC",
    "overall/tpr": "TPR (sensitivity)",
    "overall/tnr": "TNR (specificity)",
}
fig = plot_seed_points(summary, overall)
fig.suptitle("Overall performance (dots: seeds, bar: mean)", fontsize=11)
fig.savefig(FIGURES / "overall.png", dpi=150)

## 群別性能

群ごとのAUROC・TPR・TNR。陽性または陰性が20件未満のセルは指標を出さない（空欄）。
群名の下の陽性数/件数で、群の大きさを併せて読む。

In [ ]:
for metric in ("auroc", "tpr", "tnr"):
    fig = plot_group_metric(groups, metric)
    fig.savefig(FIGURES / f"groups_{metric}.png", dpi=150)

## 公平性指標

軸ごとのworst-group AUROC・AUROC gap・Eopp1（TPR gap）・Eopp0（TNR gap）・Eodds・BACC gap。
gap系は小さいほど公平。gapが縮んだときは、worstが上がったのかbestが下がったのかを前の群別図で確かめる。

In [ ]:
fairness = {}
for axis in AXES:
    for metric, name in [
        ("worst_auroc", "worst AUROC"),
        ("auroc_gap", "AUROC gap"),
        ("Eopp1", "Eopp1"),
        ("Eopp0", "Eopp0"),
        ("Eodds", "Eodds"),
        ("bacc_gap", "BACC gap"),
    ]:
        if summary[f"{axis}/{metric}"].notna().any():
            fairness[f"{axis}/{metric}"] = f"{axis}: {name}"
fig = plot_seed_points(summary, fairness, ncols=6)
fig.suptitle("Fairness metrics (lower gap is fairer)", fontsize=11)
fig.savefig(FIGURES / "fairness.png", dpi=150)

## ResNet-50との差とtest標本の雑音

比較モデル−ResNet-50を同seedで取り、3 seedで平均する。横棒は画像（=患者）単位のpaired bootstrap（1000回）の95%区間で、
各反復で同じ再標本を全モデル・全seedに当てる。小点はseedごとの差。区間が0をまたぐ差は、
このtest集合では標本雑音と区別できない。

In [ ]:
CLUSTERS = frame["image"].to_numpy()  # 1患者1枚なので画像単位で再標本化する

boot = paired_bootstrap(scores, frame, PRIMARY, CLUSTERS, n_boot=N_BOOT)
boot.to_csv(RESULTS / "paired_bootstrap.csv", index=False)
deltas = seed_deltas(summary)
deltas.to_csv(RESULTS / "paired_deltas_by_seed.csv", index=False)
headline = {"overall/auroc": "overall AUROC", "overall/bacc": "overall BACC"}
for axis in AXES:
    for metric, name in [
        ("worst_auroc", "worst AUROC"),
        ("auroc_gap", "AUROC gap"),
        ("Eopp1", "Eopp1"),
        ("Eopp0", "Eopp0"),
    ]:
        if boot.loc[boot["metric"] == f"{axis}/{metric}", "valid_draws"].min() > 0.9 * N_BOOT:
            headline[f"{axis}/{metric}"] = f"{axis}: {name}"
fig = plot_paired_deltas(boot, deltas, headline)
fig.suptitle("Difference from ResNet-50 (3-seed mean, 95% paired bootstrap CI)", fontsize=11)
fig.savefig(FIGURES / "paired_deltas.png", dpi=150)

In [ ]:
group_auroc = boot[boot["metric"].str.count("/") == 2]
group_auroc.to_csv(RESULTS / "paired_bootstrap_group_auroc.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 0.3 * len(group_auroc) + 1), layout="constrained")
for offset, (model, part) in enumerate(group_auroc.groupby("model", sort=False)):
    y = np.arange(len(part)) * 2 + offset * 0.6
    ax.errorbar(
        part["delta"],
        y,
        xerr=[part["delta"] - part["ci_low"], part["ci_high"] - part["delta"]],
        fmt="o",
        capsize=3,
        label=MODELS[model],
        color=f"C{offset}",
    )
labels = group_auroc.loc[group_auroc["model"] == group_auroc["model"].iloc[0], "metric"].str.rsplit("/", n=1).str[0]
ax.set_yticks(np.arange(len(labels)) * 2 + 0.3, labels, fontsize=8)
ax.invert_yaxis()
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Per-group AUROC: model − ResNet-50 (95% paired bootstrap CI)")
ax.legend(fontsize=8)
fig.savefig(FIGURES / "paired_deltas_group_auroc.png", dpi=150)

## 感度分析

評価集合を替えても同じ傾向かを見る。

- ほぼ黒い画像（`image_blank`、112件）を除く：Black（41/305件）に偏るので、race間の差の一部は画質の差になりうる
- FairCLIPの評価対象（`fairclip_eval_subset`、1,883件）：先行研究の数値と突き合わせるための集合

In [ ]:
sets = {
    "all 2,000 (primary)": PRIMARY,
    "blank images excluded": ~frame["image_blank"].astype(bool).to_numpy(),
    "FairCLIP eval subset": frame["fairclip_eval_subset"].astype(bool).to_numpy(),
}
rows = []
for name, mask in sets.items():
    part, _ = evaluate_all(scores, frame, mask)
    rows.append(part.assign(evaluation_set=name, n=int(mask.sum()), n_positive=int(frame.loc[mask, "target"].sum())))
sensitivity = pd.concat(rows, ignore_index=True)
sensitivity.to_csv(RESULTS / "sensitivity_by_seed.csv", index=False)
metrics = ["overall/auroc", "race/worst_auroc", "race/auroc_gap", "race/Eopp0"]
fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.2), layout="constrained")
for ax, metric in zip(axes, metrics, strict=True):
    for offset, model in enumerate(MODELS):
        stats = (
            sensitivity[sensitivity["model"] == model]
            .groupby("evaluation_set", sort=False)[metric]
            .agg(["mean", "std"])
        )
        ax.errorbar(
            np.arange(len(stats)) + (offset - 1) * 0.2,
            stats["mean"],
            yerr=stats["std"],
            fmt="o",
            capsize=2,
            label=MODELS[model],
            color=f"C{offset}",
        )
    ax.set_xticks(range(len(sets)), sets.keys(), rotation=15, fontsize=8)
    ax.set_title(metric, fontsize=10)
axes[0].legend(fontsize=8)
fig.savefig(FIGURES / "sensitivity.png", dpi=150)

## まとめ

**観察事実**（主比較：test全2,000件、3 seed、差は同seed差の平均と画像単位bootstrapの95%区間）

- 全体AUROCは3モデルとも0.81前後で、ResNet-50との差は区間が0をまたぐ（HyperAdapt +0.004、Spatial LoRA −0.004）。
- 群別AUROCとrace軸のworst-group AUROC・AUROC gapにも差が無い。blank画像を除いても、FairCLIPの評価対象でも同じ。
- 閾値に依存する指標では差が出る。HyperAdaptはrace軸のEopp0が+0.14［0.05, 0.19］、Eoddsが+0.09悪化した。
  BlackのTNRが0.58（ResNet-50 0.64）に下がり、WhiteのTNRは0.79（同0.71）に上がる。
- 全体のTPR・TNRは、条件付きモデルで特異度側へ動く。ただしResNet-50の動作点はseedでばらつく（TNRのSD 0.13）。

**解釈候補**

- FairVLMedでは、属性条件付けは順位付けの性能も群間のAUROC差も変えていない。
- HyperAdaptの変化は、raceごとの事前確率の違い（trainの陽性率：Black 65%、他は48%）を出力に反映し、
  群ごとに判定を陽性側・陰性側へ寄せた形と整合する。群内AUROCは事前確率に依らないので変わらない。
- CheXpertと同じく、hypernetworkの効果はAUROCベースの公平性には見えない。データセット固有の問題という説明は、
  この結果だけでは支持されない。

**追加確認**

- 群ごとの予測確率の平均と較正（reliability）を見て、事前確率の反映という解釈を確かめる
- 動作点を揃えた比較（群をまたいで同じ特異度に合わせたTPR）で、閾値依存の差が消えるか